In [0]:
import os
import sys
from pathlib import Path

# En Serverless, indicar la raíz absoluta del proyecto completo en /Workspace/...
# Puede fijarse aquí o mediante RISKVISION_PROJECT_ROOT; no debe apuntar a notebooks/.
PROJECT_ROOT_OVERRIDE = os.environ.get("RISKVISION_PROJECT_ROOT")
if PROJECT_ROOT_OVERRIDE:
    project_root = Path(PROJECT_ROOT_OVERRIDE).expanduser()
    if not project_root.is_absolute():
        raise ValueError("RISKVISION_PROJECT_ROOT debe ser una ruta absoluta")
    project_root = project_root.resolve()
else:
    # Compatibilidad local: iniciar desde la raíz o desde notebooks/.
    candidates = (Path.cwd().resolve(), Path.cwd().resolve().parent)
    project_root = next((p for p in candidates if (p / "src" / "proyecto").is_dir()), None)
if project_root is None or not (project_root / "src" / "proyecto" / "config.py").is_file():
    raise RuntimeError("No se encuentra src/proyecto: configura RISKVISION_PROJECT_ROOT con la raíz absoluta")
source_dir = str(project_root / "src")
if source_dir not in sys.path:
    sys.path.insert(0, source_dir)
import proyecto
if Path(proyecto.__file__).resolve().parent != project_root / "src" / "proyecto":
    raise RuntimeError("El kernel tiene cargada otra copia de proyecto; reinícialo y revisa la raíz")
print("Raíz verificada:", project_root)
print("Módulos:", Path(proyecto.__file__).resolve().parent)
print("Python:", sys.version.split()[0], "| plataforma:", sys.platform)

# El kernel de Serverless precarga paquetes del runtime (pandas 2.3.3,
# typing_extensions antiguo, etc.) en sys.modules ANTES de activar el entorno
# del proyecto. Eso impide importar las versiones del entorno (train.joblib
# exige pandas 3.0.6; anyio exige typing_extensions nuevo) y el diagnóstico
# abortaría. Purga y recarga: todo módulo precargado desde
# /databricks/python/lib cuya distribución exista también en el entorno del
# proyecto se reimporta desde ahí. No se alteran métodos internos ni se
# instalan paquetes. Efecto por sesión; reiniciar la sesión lo deshace.
import importlib.metadata as _im
from pathlib import Path as _Path
_venv_sp = next((_Path(_p) for _p in sys.path
                 if "envs" in _p and _p.rstrip("/").endswith("site-packages")), None)
if _venv_sp is not None and _venv_sp.is_dir():
    _nombres_venv = {(_q.name[:-3] if _q.name.endswith(".py") else _q.name)
                     for _q in _venv_sp.iterdir()}
    _a_purgar = sorted(_n for _n, _m in list(sys.modules.items())
                       if getattr(_m, "__file__", None)
                       and "/databricks/python/lib/" in _m.__file__
                       and _n.split(".")[0] in _nombres_venv)
    for _n in _a_purgar:
        del sys.modules[_n]
    if _a_purgar:
        print("Módulos del runtime reemplazados por los del entorno del proyecto:",
              len(_a_purgar), "->", sorted({_n.split(".")[0] for _n in _a_purgar}))
        _p_instalado = _im.version("pandas")
        import pandas
        assert pandas.__version__ == _p_instalado, "la recarga no tomó la versión del entorno"
from proyecto.runtime import diagnose_environment
workspace_environment = ("DATABRICKS_RUNTIME_VERSION" in os.environ
                         or project_root.as_posix().startswith(("/Workspace/", "/Volumes/")))
ENVIRONMENT_MODE = os.environ.get("RISKVISION_ENVIRONMENT_MODE",
                                  "serverless" if workspace_environment else "local")
if ENVIRONMENT_MODE not in {"local", "serverless"}:
    raise ValueError("RISKVISION_ENVIRONMENT_MODE debe ser local o serverless")
if workspace_environment and ENVIRONMENT_MODE != "serverless":
    raise ValueError("En Databricks se exige el diagnóstico serverless")
ENVIRONMENT_DIAGNOSTIC = diagnose_environment(
    project_root / "requirements-serverless-06.txt", strict=ENVIRONMENT_MODE == "serverless")
print("Diagnóstico previo a datos:", ENVIRONMENT_DIAGNOSTIC)

from proyecto.models.persistence import (PersistenceControls, load_bundle, verify_bundle,
    save_bundle, train_final_reference, synthetic_persistence_check)
from proyecto.tracking.model_registry import (publish_bundle_artifacts,
    register_definitive_model, promote_model_version)
from proyecto.models.candidates import LGBM_PARAMS
from proyecto.features.build import PACKAGE_E
from proyecto.tracking.reporting import code_revision
from copy import deepcopy
import tempfile
import json

# Ajuste final y publicación de evidencia autorizados por la usuaria en esta
# sesión; registro definitivo autorizado tras la aprobación definitiva; solo la
# promoción (alias champion) sigue bloqueada como decisión separada.
ALLOW_FINAL_REAL_TRAINING = True
ALLOW_DEFINITIVE_REGISTRATION = True
ALLOW_MODEL_PROMOTION_OR_APPROVAL = False
ALLOW_REAL_ARTIFACT_PUBLICATION = True
RUN_SYNTHETIC_CHECK = False
CONTROLS = PersistenceControls(
    train_real=ALLOW_FINAL_REAL_TRAINING,
    register_definitive=ALLOW_DEFINITIVE_REGISTRATION,
    promote_or_approve=ALLOW_MODEL_PROMOTION_OR_APPROVAL,
    publish_real_artifacts=ALLOW_REAL_ARTIFACT_PUBLICATION)


# 08 · Registro y persistencia — RiskVision, Grupo 6

Se conserva el recorrido académico: conexión, procedencia, registro, alias, recuperación y constancia. **Tracking** guarda evidencia de una ejecución; **Model Registry** crea versiones de modelos. Guardar un Joblib no equivale a registrar, aprobar ni promover.

La referencia es LightGBM original con paquete E, todavía **PROVISIONAL**. No existe evaluación real completa de 07 ni modelo definitivo aprobado. No se atribuyen resultados de otra rama a esta implementación.

Todos los controles reales están desactivados. Esta entrega comprueba infraestructura con datos sintéticos, sin leer datasets ni publicar en Databricks. Configurar Environment con requirements-serverless-06.txt y ejecutar desde la primera celda; no hay dependencias nuevas. Las versiones efectivas viajan con cada artefacto y se verifican al recuperarlo.


## 0 · Conexión y trazabilidad

La configuración automática local/Databricks se reutiliza, pero solo se conecta si existe autorización de publicación. El experimento debe indicarse explícitamente. En Databricks, Model Registry usa Unity Catalog: indicar catalog.schema.model y comprobar permisos, catálogo y esquema. No crear destinos predeterminados ni sobrescribir experimentos.


In [0]:
# Experimento NUEVO de evidencia provisional 08; no se reutiliza ninguno anterior.
from datetime import datetime as _dt, timezone as _tz
from uuid import uuid4 as _uuid4
EXPERIMENT_NAME = "/Users/estefaniacg1995@gmail.com/riskvision-08-" + _dt.now(_tz.utc).strftime("%Y%m%dT%H%M%SZ") + "-" + _uuid4().hex
REGISTERED_MODEL_NAME = None  # catalog.schema.model en Unity Catalog
EXPERIMENT_ID = None
TRACKING_RECEIPT = REGISTRY_RECEIPT = PROMOTION_RECEIPT = None
if CONTROLS.publish_real_artifacts:
    if not EXPERIMENT_NAME:
        raise ValueError("Falta la ruta completa del experimento autorizado")
    import mlflow
    from proyecto.tracking.mlflow_utils import configure_mlflow, set_experiment
    configure_mlflow()
    experiment_path = set_experiment(EXPERIMENT_NAME)
    experiment = mlflow.get_experiment_by_name(experiment_path)
    if experiment is None:
        raise RuntimeError("No se recuperó el experimento por su ruta completa")
    EXPERIMENT_ID = experiment.experiment_id
REVISION = code_revision(project_root)
print("Controles:", CONTROLS, "| revisión:", REVISION)


## 1 · Modelo y decisión de procedencia

No cargar un supuesto final_pipeline.joblib ni fabricar métricas. La aprobación, configuración congelada y huella del entrenamiento deben proceder de una revisión explícita de 07. El ajuste final reutiliza su función de entrenamiento; por defecto ni siquiera se accede a datos.

Para recuperar un paquete real se requiere procedencia confiable, una huella externa del manifiesto y un entorno compatible. SHA-256 detecta cambios, pero no autentica al remitente. Nunca cargar Joblib de origen desconocido.

El contrato 07→08 proviene de decision.py. Se comprueba su huella externa, autorización humana de ajuste final y configuración congelada. La aprobación definitiva se aporta por separado, ligada al mismo informe y huella de configuración; no se deduce de métricas.


In [0]:
from proyecto.models.decision import read_handoff, approve_for_inference, adopted_experiment, selected_validation_metrics
from proyecto.models.validation import load_training
from proyecto.data.split import make_xy
REFERENCE_DECISION = {"status": "provisional", "model": "LightGBM original",
    "features": list(PACKAGE_E), "params": deepcopy(LGBM_PARAMS),
    "own_real_optimization_executed": False, "antecedent": adopted_experiment()}
# Contrato 07→08 generado en la sesión de validación real de la etapa 07
# (traspaso autorizado por Estefanía). Huella externa verificada al leer.
HANDOFF_PATH = str(project_root / "reports" / "07_handoff_to_08.json")
HANDOFF_SHA256 = "8094a9474947d314c72891eac781aedb75247a0f3e6a57e06acc5e8749e2717b"
TRUST_HANDOFF_SOURCE = True
# Aprobación definitiva externa autorizada por la usuaria en esta sesión, ligada
# al informe 07 (report_id) y a la configuración congelada del contrato 07→08.
APPROVE_DEFINITIVE_DECISION = True
DEFINITIVE_APPROVAL_RECORD = {
    "approved": True, "approved_by": "Estefanía",
    "approved_at": _dt.now(_tz.utc).isoformat(),
    "validation_report_id": "e479fb62f1434131890fe48799f9fe6c",
    "configuration_sha256": "1e52b4db6e8561d8e0765233c0664dd8cc5c3905d25cc6e83f705de7c2fa017b"}
HANDOFF_DECISION = None
MODEL_DECISION = None
TRAINING_FRAME = FITTED_MODEL = TRAINING_INPUTS = None
# Directorio NUEVO, persistente y fuera de Git (destino autorizado en esta sesión).
BUNDLE_OUTPUT = Path("/Workspace/Users/estefaniacg1995@gmail.com/riskvision/bundles/08-candidate-"
                     + _dt.now(_tz.utc).strftime("%Y%m%dT%H%M%SZ"))
BUNDLE_PATH = EXPECTED_MANIFEST_SHA256 = None
TRUST_BUNDLE_SOURCE = False
BUNDLE = None
if HANDOFF_PATH is not None:
    HANDOFF_DECISION = read_handoff(HANDOFF_PATH, trusted=TRUST_HANDOFF_SOURCE, expected_sha256=HANDOFF_SHA256)
    MODEL_DECISION = HANDOFF_DECISION  # puede seguir provisional; no confundir con aprobación definitiva
if APPROVE_DEFINITIVE_DECISION:
    if HANDOFF_DECISION is None:
        raise PermissionError("Falta contrato verificado de 07")
    MODEL_DECISION = approve_for_inference(HANDOFF_DECISION,
        approved=True, approval=DEFINITIVE_APPROVAL_RECORD)
if CONTROLS.train_real:
    if MODEL_DECISION is None:
        raise PermissionError("El ajuste final exige el contrato autorizado de 07")
    training_path = Path(os.environ.get("RISKVISION_TRAIN_PATH", str(project_root / "data/processed/train.joblib")))
    if not training_path.is_absolute():
        raise ValueError("La ruta de entrenamiento debe ser absoluta")
    TRAINING_FRAME = load_training(training_path, expected_sha256=MODEL_DECISION["training_sha256"])
    FITTED_MODEL = train_final_reference(TRAINING_FRAME, decision=MODEL_DECISION, controls=CONTROLS)
    TRAINING_INPUTS, _ = make_xy(TRAINING_FRAME)


## 2 · Qué se persiste y qué se registra

El paquete contiene pipeline.joblib (variables E, preprocesamiento, codificación, estimador ajustado y, si corresponde, calibración), input_schema.json, metadata.json y manifest.json. Cada directorio es nuevo: no se sobrescribe una versión anterior.

Los metadatos incluyen nombre/versión, estado, parámetros efectivos, procedencia y decisión, rama/commit, fecha UTC, dependencias, huellas, experimento y métricas solo cuando existan. La versión local no es una versión del Registry.

El contrato acepta exactamente los 23 predictores originales **ya limpios**, permite faltantes para los imputadores aprendidos y facturas negativas plausibles. Rechaza ID, objetivo, derivadas externas, categorías inválidas, infinitos y dominios incompatibles. La limpieza previa debe reutilizar las reglas del proyecto. No se crea un umbral operativo ni se aprueba la hipótesis económica 5:1.

Publicar evidencia provisional en Tracking requiere permiso; registrar una versión definitiva exige además aprobación documentada. MLflow usa el pipeline completo, firma, ejemplo explícito y código importable de proyecto. Joblib ya soporta el selector importable; cloudpickle se usa como formato del flavor MLflow, no como sustitución de aquella corrección.


In [0]:
BUNDLE_RECEIPT = None
if CONTROLS.publish_real_artifacts and FITTED_MODEL is not None:
    if BUNDLE_OUTPUT is None or TRAINING_INPUTS is None or MODEL_DECISION is None:
        raise ValueError("Faltan destino nuevo, entradas del ajuste y decisión revisada")
    BUNDLE_RECEIPT = save_bundle(FITTED_MODEL, TRAINING_INPUTS, BUNDLE_OUTPUT,
        model_name="riskvision_lightgbm_e", model_version="candidate-08",
        decision=MODEL_DECISION, controls=CONTROLS, experiment_id=EXPERIMENT_ID, revision=REVISION,
        validation_metrics=selected_validation_metrics(MODEL_DECISION))
    BUNDLE_PATH = BUNDLE_RECEIPT["directory"]
    EXPECTED_MANIFEST_SHA256 = BUNDLE_RECEIPT["manifest_sha256"]
    TRUST_BUNDLE_SOURCE = True
if BUNDLE_PATH is not None:
    if not TRUST_BUNDLE_SOURCE or not EXPECTED_MANIFEST_SHA256:
        raise PermissionError("Se requieren origen confiable y huella externa")
    BUNDLE = load_bundle(BUNDLE_PATH, trusted=True, expected_manifest_sha256=EXPECTED_MANIFEST_SHA256)
if CONTROLS.publish_real_artifacts:
    if BUNDLE is None:
        raise ValueError("Falta un paquete real verificado")
    TRACKING_RECEIPT = publish_bundle_artifacts(BUNDLE, experiment_id=EXPERIMENT_ID, controls=CONTROLS)
# Clientes de ejemplo explícitamente autorizados por la usuaria en esta sesión:
# 2 clientes del ajuste final (TRAINING_INPUTS, datos aprobados del contrato 07→08).
# Recarga del módulo en esta sesión: corrección de claves de tag de versión (sin ".").
import importlib as _il
import proyecto.tracking.model_registry as _model_registry
_il.reload(_model_registry)
register_definitive_model = _model_registry.register_definitive_model
INPUT_EXAMPLE = None if TRAINING_INPUTS is None else TRAINING_INPUTS.head(2)
REGISTERED_MODEL_NAME = "workspace.default.riskvision_credito"  # catalog.schema.model
if CONTROLS.register_definitive:
    if BUNDLE is None:
        raise PermissionError("Registro bloqueado: falta un modelo aprobado")
    REGISTRY_RECEIPT = register_definitive_model(BUNDLE, INPUT_EXAMPLE,
        registered_name=REGISTERED_MODEL_NAME, experiment_id=EXPERIMENT_ID,
        source_code_dir=project_root / "src/proyecto", controls=CONTROLS)


## 3 · Alias: promoción separada

champion es un alias, no una prueba de calidad. Nunca buscar la versión máxima ni promocionar automáticamente. Solo se permite asignar un alias a la versión exacta devuelta por el registro, con aprobación externa y comprobación de su huella. Este módulo no aprueba modelos.


In [0]:
if CONTROLS.promote_or_approve:
    if BUNDLE is None or REGISTRY_RECEIPT is None:
        raise PermissionError("Promoción bloqueada: falta paquete aprobado y recibo exacto")
    PROMOTION_RECEIPT = promote_model_version(BUNDLE, REGISTRY_RECEIPT, controls=CONTROLS)


## 4 · Round-trip y prueba sintética

La prueba opcional crea clientes sintéticos, ajusta la referencia original, guarda y recupera el pipeline completo y compara clases y probabilidades. Todos los archivos están en un directorio temporal eliminado al terminar. No es evidencia de rendimiento real.

Las pruebas automatizadas comprueban además recuperación desde un proceso Python nuevo, alteraciones de archivos, esquema, controles y simulaciones de MLflow. Las versiones del artefacto deben reproducirse; un hash no elimina los riesgos de pickle.


In [0]:
SYNTHETIC_EVIDENCE = None
if RUN_SYNTHETIC_CHECK:
    with tempfile.TemporaryDirectory(prefix="riskvision08-") as directory:
        SYNTHETIC_EVIDENCE = synthetic_persistence_check(Path(directory) / "bundle")
    print("Prueba sintética (directorio ya eliminado):", SYNTHETIC_EVIDENCE)
else:
    print("Prueba sintética opcional desactivada; no se generaron artefactos")


## 5 · Constancia y continuidad hacia 09

El informe versionado reports/08_registro.md documenta el protocolo, no inventa resultados reales. Esta constancia queda en memoria; cada publicación autorizada crea un run nuevo y conserva sus huellas. No se reemplazan informes de ejecuciones anteriores.

Para 09 se reutiliza load_bundle(..., trusted=True, expected_manifest_sha256=..., require_approved=True) y su predict_proba/predict con validación. El paquete también expone classes_ y schema para predict_with_scores. Mantener la política de umbral separada; la clase del estimador no es una decisión económica aprobada. El notebook 09 no se modifica en esta fase.


In [0]:
REPORT_08 = {"stage": "08",
    "status": BUNDLE.metadata["status"] if BUNDLE is not None else (MODEL_DECISION or REFERENCE_DECISION)["status"],
    "revision": REVISION, "decision": MODEL_DECISION or REFERENCE_DECISION,
    "controls": {"train_real": CONTROLS.train_real, "register_definitive": CONTROLS.register_definitive,
        "promote_or_approve": CONTROLS.promote_or_approve, "publish_real_artifacts": CONTROLS.publish_real_artifacts},
    "handoff_sha256": HANDOFF_SHA256, "synthetic_evidence": SYNTHETIC_EVIDENCE,
    "tracking": TRACKING_RECEIPT, "registry": REGISTRY_RECEIPT, "promotion": PROMOTION_RECEIPT,
    "validation_report_id": None if MODEL_DECISION is None else MODEL_DECISION["handoff"]["validation_report"]["report_id"],
    "real_validation_metrics": None if MODEL_DECISION is None else MODEL_DECISION["handoff"]["validation_report"]["internal_evaluation"],
    "real_results": "pendientes" if BUNDLE is None else "artefacto verificado; calidad según informe 07, no reevaluada aquí"}
print(json.dumps(REPORT_08, ensure_ascii=False, indent=2))


### Pendiente antes de continuar con resultados reales

Completar y revisar la evaluación 07; congelar parámetros, calibración y umbral solo con entrenamiento; documentar aprobación y datos. La prueba original ya fue explorada y no constituye una evaluación final completamente independiente. Preparar permisos y almacenamiento persistente en Databricks, comprobar la carga del paquete en una sesión nueva y revisar firmas/dependencias antes de autorizar cualquier publicación. No se ejecutó esta integración de Registry en Serverless ni se declaró un modelo definitivo.


In [0]:
# Evidencia: serviciabilidad del artefacto registrado en entorno aislado
# (realizada tras el registro de workspace.default.riskvision_credito v2;
# los tags de versión se aplicaron con claves con guion bajo tras detectar que
# Unity Catalog no admite "." en claves de tag de versión).
import shutil
print("uv disponible:", shutil.which("uv"))
import mlflow
example = INPUT_EXAMPLE.loc[:, BUNDLE.schema["input_columns"]].copy()
for name in example.select_dtypes(include="category"):
    example[name] = example[name].astype(float)
result = mlflow.models.predict(
    model_uri="models:/workspace.default.riskvision_credito/2",
    input_data=example, env_manager="uv")
print("Predicción en entorno aislado OK:")
print(result)

In [0]:
# Prueba de humo del endpoint riskvision-credito con el mismo ejemplo del
# registro (predicciones del pipeline servido en Model Serving).
from databricks.sdk import WorkspaceClient
_ws = WorkspaceClient()
_ep = _ws.serving_endpoints.get("riskvision-credito")
print("Endpoint:", _ep.name, "| route_optimized:", _ep.route_optimized,
      "| ready:", _ep.state.ready)
_resp = _ws.serving_endpoints.query(name="riskvision-credito",
    dataframe_records=example.to_dict(orient="records"))
print("Respuesta del endpoint:", _resp.predictions)